# 02 — Model Training

Trains all five models reported in the paper (Table 2):

- **RF (EASD)** — Random Forest on hand-engineered topographic features
- **RF (AE)** — Random Forest on raw 64-dim AlphaEarth embeddings
- **MLP (AE)** — Multi-layer perceptron on raw AlphaEarth embeddings (selected model)
- **CNN 3×3 (AE)** — Patch-based CNN, 3×3 spatial context
- **CNN 5×5 (AE)** — Patch-based CNN, 5×5 spatial context

All models use Strategy 2 (geographic hold-out, R3 withheld for validation). All logic lives in `glacier_melt.sampling`, `glacier_melt.models`, and `glacier_melt.train`; this notebook only loads data, calls those functions, and saves results.

In [ ]:
from pathlib import Path
import joblib
import torch
import numpy as np

from glacier_melt.sampling import (
    load_parquet_regions, strategy2_split, EASD_COLS, AE_COLS, LABEL_COL,
)
from glacier_melt.models import build_model
from glacier_melt.train import (
    train_rf, sample_for_rf,
    build_pixel_tensors, build_patch_tensors,
    save_tensors, load_tensors,
    make_loaders, train_model, predict,
)

DATA_DIR = Path("../data/Training_Data_2017")
TENSOR_DIR = Path("../data/Tensors")
MODEL_DIR = Path("../data/Final_Models")
TENSOR_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_REGION = "r3"
TRAIN_REGIONS = ["r1a", "r1b", "r2"]

## Load data and apply geographic hold-out split

In [ ]:
df = load_parquet_regions(DATA_DIR)
train_df, val_df = strategy2_split(df, validation_region=VALIDATION_REGION)

## Random Forest — EASD baseline

Trained on hand-engineered topographic features (elevation, aspect, slope, edge distance), reproducing Andriychenko (2024)'s baseline approach for direct comparison.

In [ ]:
rf_easd_sample, _ = sample_for_rf(train_df, n_samples=2_000_000)

X_train_easd = rf_easd_sample[EASD_COLS].values
y_train_easd = rf_easd_sample[LABEL_COL].values

rf_easd = train_rf(X_train_easd, y_train_easd)
joblib.dump(rf_easd, MODEL_DIR / "rf_easd.joblib")

## Random Forest — raw AlphaEarth embeddings

Trained on the full 64-dimensional AlphaEarth embedding vector, testing whether the RF architecture itself benefits from richer features (consistent with Grinsztajn et al. 2022's finding that tree-based models remain competitive with neural networks on tabular data).

In [ ]:
rf_ae_sample, _ = sample_for_rf(train_df, n_samples=2_000_000)

X_train_ae = rf_ae_sample[AE_COLS].values
y_train_ae = rf_ae_sample[LABEL_COL].values

rf_ae = train_rf(X_train_ae, y_train_ae)
joblib.dump(rf_ae, MODEL_DIR / "rf_ae.joblib")

## MLP — selected deployment model

Single-pixel classifier on raw AlphaEarth embeddings. Selected for deployment over the CNN variants due to the smallest train/validation overlap gap (1.58%), indicating the most robust cross-regional generalisation (see Section 3.4.2 of the paper).

In [ ]:
mlp_train_path = TENSOR_DIR / "mlp_train.pt"
mlp_val_path = TENSOR_DIR / "mlp_val.pt"

if mlp_train_path.exists():
    mlp_train_tensors = load_tensors(mlp_train_path)
    mlp_val_tensors = load_tensors(mlp_val_path)
else:
    mlp_train_tensors = build_pixel_tensors(train_df)
    mlp_val_tensors = build_pixel_tensors(val_df)
    save_tensors(mlp_train_tensors, mlp_train_path)
    save_tensors(mlp_val_tensors, mlp_val_path)

In [ ]:
from torch.utils.data import TensorDataset

melt_rate = mlp_train_tensors.tensors[1].cpu().mean().item()
pos_weight = torch.tensor((1 - melt_rate) / melt_rate)

from glacier_melt.train import GPUDataLoader
mlp_train_loader = GPUDataLoader(mlp_train_tensors, batch_size=65536, shuffle=True)
mlp_val_loader = GPUDataLoader(mlp_val_tensors, batch_size=65536, shuffle=False)

mlp_model = build_model("mlp", dropout=0.5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp_model.to(device)

best_val_auc = train_model(
    mlp_model, mlp_train_loader, mlp_val_loader, pos_weight,
    n_epochs=20, patience=5,
    save_path=MODEL_DIR / "MLP_AE64_R3HOLDOUT.pt",
)

## CNN — 3×3 spatial patch

In [ ]:
cnn3_train_path = TENSOR_DIR / "cnn3x3_train.pt"
cnn3_val_path = TENSOR_DIR / "cnn3x3_val.pt"

if cnn3_train_path.exists():
    cnn3_train_tensors = load_tensors(cnn3_train_path)
    cnn3_val_tensors = load_tensors(cnn3_val_path)
else:
    cnn3_train_tensors = build_patch_tensors(train_df, patch_size=3)
    cnn3_val_tensors = build_patch_tensors(val_df, patch_size=3)
    save_tensors(cnn3_train_tensors, cnn3_train_path)
    save_tensors(cnn3_val_tensors, cnn3_val_path)

cnn3_train_loader = GPUDataLoader(cnn3_train_tensors, batch_size=65536, shuffle=True)
cnn3_val_loader = GPUDataLoader(cnn3_val_tensors, batch_size=65536, shuffle=False)

cnn3_model = build_model("cnn3x3", dropout=0.5).to(device)

train_model(
    cnn3_model, cnn3_train_loader, cnn3_val_loader, pos_weight,
    n_epochs=20, patience=5,
    save_path=MODEL_DIR / "CNN3x3_AE64_R3HOLDOUT.pt",
)

## CNN — 5×5 spatial patch

Not selected for deployment (larger train/validation gap, 11.05%), but trained for the model comparison in Table 2.

In [ ]:
cnn5_train_path = TENSOR_DIR / "cnn5x5_train.pt"
cnn5_val_path = TENSOR_DIR / "cnn5x5_val.pt"

if cnn5_train_path.exists():
    cnn5_train_tensors = load_tensors(cnn5_train_path)
    cnn5_val_tensors = load_tensors(cnn5_val_path)
else:
    cnn5_train_tensors = build_patch_tensors(train_df, patch_size=5)
    cnn5_val_tensors = build_patch_tensors(val_df, patch_size=5)
    save_tensors(cnn5_train_tensors, cnn5_train_path)
    save_tensors(cnn5_val_tensors, cnn5_val_path)

cnn5_train_loader = GPUDataLoader(cnn5_train_tensors, batch_size=65536, shuffle=True)
cnn5_val_loader = GPUDataLoader(cnn5_val_tensors, batch_size=65536, shuffle=False)

cnn5_model = build_model("cnn5x5", dropout=0.5).to(device)

train_model(
    cnn5_model, cnn5_train_loader, cnn5_val_loader, pos_weight,
    n_epochs=20, patience=5,
    save_path=MODEL_DIR / "CNN5x5_AE64_R3HOLDOUT.pt",
)

## Save Peru-wide predictions for all models

Used by `03_evaluation.ipynb` to reproduce Table 2 and the per-glacier analysis without re-running training.

In [ ]:
import pandas as pd

PREDICTIONS_DIR = Path("../data/Predictions")
PREDICTIONS_DIR.mkdir(exist_ok=True)

full_pixel_tensors = build_pixel_tensors(df)
full_3x3_tensors = build_patch_tensors(df, patch_size=3)
full_5x5_tensors = build_patch_tensors(df, patch_size=5)

df_predictions = df[["lon", "lat", "region", LABEL_COL]].copy()
df_predictions["rf_easd_prob"] = rf_easd.predict_proba(df[EASD_COLS].values)[:, 1]
df_predictions["rf_ae_prob"] = rf_ae.predict_proba(df[AE_COLS].values)[:, 1]
df_predictions["mlp_prob"] = predict(mlp_model, full_pixel_tensors)
df_predictions["cnn3x3_prob"] = predict(cnn3_model, full_3x3_tensors)
df_predictions["cnn5x5_prob"] = predict(cnn5_model, full_5x5_tensors)

df_predictions.to_parquet(
    PREDICTIONS_DIR / "predictions_full_peru_r3holdout.parquet", index=False
)
print(f"Saved predictions for {len(df_predictions):,} pixels")